RQ3 -----------------------------------------------

Random Forest

Facemask wearing - Omicron wave  

Dataset with state and covid 7 day rolling cases and deaths are considered here for the analysis.

In [1]:
# import libraries
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
import joblib


omicron_train_facemask = pd.read_csv("omicron_train_facemask.csv")
omicron_test_facemask = pd.read_csv("omicron_test_facemask.csv")

omicron_train_facemask.columns

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'household_size',
       'Wellbeing', 'Perceived Severity', 'Perceived Susceptibility',
       'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment'

Random Forest

In [2]:
# remove the target variables and identifiers
drop_cols = ['RecordNo', 'Date', 'face_mask_scale', 'face_mask_binary','general_protective_behavior_scale',
                'general_protective_behavior_binary','mandate_start_date', 'wave'] # wave 


cv = StratifiedKFold( # 5 fold cross validation
        n_splits=5,
        shuffle=True,
        random_state=42
)

  
cv_rf = Pipeline([   # standard scaler is not important in RF as RF makes decisions on the order of them not the scale
            ('ros', RandomOverSampler(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42,n_jobs=-1))
    ])   


params = { # apply the parameters to the rf step of the pipeline - rf__n_estimators  double underscore
        
            'rf__n_estimators': [250], 
            'rf__max_depth': [5,7],
            'rf__min_samples_split': [2,10],  # 2,10
            'rf__min_samples_leaf': [1,5],    # 1,5
            'rf__max_features': ['sqrt','log2']  # important for RF
    }



# predictors
x_train = omicron_train_facemask.drop(columns=drop_cols)
x_test = omicron_test_facemask.drop(columns=drop_cols)

x_train = x_train.astype(float)  # converting boolean to float 
x_test = x_test.astype(float)

print(x_train.columns)
print(x_train.shape)



# target variable 

y_train = omicron_train_facemask["face_mask_binary"]
y_test = omicron_test_facemask["face_mask_binary"]





# RF model ---------------------------------------------------------


grid = GridSearchCV( # tunes parameters
            cv_rf,
            params,
            cv=cv,
            scoring={  # evaluates all metrics
            'roc_auc': 'roc_auc',
            'accuracy': 'accuracy',
            'f1': 'f1'},
            refit='roc_auc',  # choose the best model
            return_train_score=False,

            n_jobs=-1 # use all available CPU cores for parallel processing
        )

grid.fit(x_train, y_train)

best_rf = grid.best_estimator_   # best model

best_parameters = grid.best_params_   # best hyper parameters

joblib.dump(best_parameters, f"RQ3_facemask_omicron_RF_bestParameters_rolling.pkl")

results_rf = pd.DataFrame(grid.cv_results_)

pred = best_rf.predict(x_test)
prob = best_rf.predict_proba(x_test)[:,1]


accuracy = round(accuracy_score(y_test, pred),4)
roc_auc = round(roc_auc_score(y_test, prob),4)
f1 = round(f1_score(y_test, pred),4)


print("Accuracy:", accuracy)
print("ROC AUC:", roc_auc)
print("F1:", f1)


joblib.dump(best_rf, f"RQ3_facemask_omicron_RF_rolling.pkl") # best model 
results_rf.to_csv(f"RQ3_facemask_omicron_RF_rolling_results.csv", index=False)


Index(['Non-household contacts', 'age', 'household_size', 'Wellbeing',
       'Perceived Severity', 'Perceived Susceptibility',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_period', 'Isolate if unwell_Not sure',
       'Isolate if unwell_Yes', 'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment's response_Don't 